Import needed modules.

In [8]:
import requests
import os

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import zipfile

from tqdm.notebook import tqdm

import huggingface_hub

Define some parameters.

In [2]:
# time interval for the dataset
YEAR: int = 2025
MONTH: int = 7
DAY_FROM: int = 1   # first day considered
DAY_TO: int = 7     # last day considered

# download
SOURCE: str = "http://aisdata.ais.dk/"
RAW_PATH: str = "dataset/raw"
PARQUET_PATH: str = "dataset/parquet"

# upload
REPOSITORY_ID: str = "andsanv/ais-tracks"

Download raw, zip files from source.

In [ ]:
# download all zips for each required day
for day in tqdm(range(DAY_FROM, DAY_TO + 1), desc="Download"):
    # build url
    filename: str = f"aisdk-{YEAR}-{MONTH:02d}-{day:02d}.zip"
    url: str = SOURCE + filename

    # create folder
    os.makedirs(RAW_PATH, exist_ok=True)
    filename = RAW_PATH + '/' + filename

    # download file
    with requests.get(url, stream=True) as r:
        with open(filename, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):  # 1 MB chunks
                if chunk:
                    f.write(chunk)

Convert to parquet format.

In [ ]:
# create target directory
os.makedirs("dataset/parquet", exist_ok=True)

# iterate over days
for day in tqdm(range(DAY_FROM, DAY_TO + 1), desc="Conversion"):
    raw_path: str = f"{RAW_PATH}/aisdk-{YEAR}-{MONTH:02d}-{day:02d}.zip"
    parquet_path: str = f"{PARQUET_PATH}/aisdk-{YEAR}-{MONTH:02d}-{day:02d}.parquet"

    # open .zip file
    with zipfile.ZipFile(raw_path) as z:
        # find the .csv file inside the .zip and open it
        csv_name: str = [n for n in z.namelist() if n.endswith(".csv")][0]

        with z.open(csv_name) as csvfile:
            # initialize objects
            parquet_writer = None; chunksize: int = 1_000_000

            # stream the .csv file in chunks
            for chunk in pd.read_csv(csvfile, chunksize=chunksize):
                table = pa.Table.from_pandas(chunk)

                if parquet_writer is None:
                    parquet_writer = pq.ParquetWriter(parquet_path, table.schema)

                parquet_writer.write_table(table)

            if parquet_writer:
                parquet_writer.close()

Upload .parquet files to HuggingFace.

In [ ]:
# login to HuggingFace and retrieve apis
huggingface_hub.login()
api = huggingface_hub.HfApi()

# find files
files = sorted([f for f in os.listdir(PARQUET_PATH) if f.endswith(".parquet")])

for file_name in files:
    # define paths
    local_path = f"{PARQUET_PATH}/{file_name}"
    remote_path = file_name  # same name in the repo

    # upload
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=remote_path,
        repo_id=REPOSITORY_ID,
        repo_type="dataset"
    )

print("all Parquet files uploaded to HuggingFace.")